In [ ]:
import glob
import ast
import time
import numpy as np
import pandas as pd
from tqdm import tqdm 
import cv2
from PIL import Image
import re
import joblib
import matplotlib.pyplot as plt
import gc
import zipfile

from tqdm import tqdm
import shutil

In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Verify GPU

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch_type = torch.float32 if device.type == "cuda" else torch.float16
device, torch_type

In [ ]:
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### Loading package

In [ ]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[2]
sys.path.append(str(repo_path))

In [ ]:
from py.utils import verifyDir,verifyFile

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

DATA_PATH = os.getenv('DATA_PATH')
MODEL_PATH = os.getenv('MODEL_PATH')
DATA_PATH, MODEL_PATH

In [ ]:
MODEL_NAME="OneFormer_Swin_Large"
SEG_DATASET="ade20k" # cityscapes

In [ ]:
ADE20K_DIR = f"{DATA_PATH}{SEG_DATASET}/"
QSCORE_PATH=f"{DATA_PATH}pp2/Qscores/"
IMAGES_PATH = f"{DATA_PATH}pp2/images/"
SEGMENT_DIR = f"{DATA_PATH}pp2/segmentations/{SEG_DATASET}/{MODEL_NAME}/"

In [ ]:
verifyDir(SEGMENT_DIR)

### Loading data

In [ ]:
%%time
data_df = pd.read_csv(f"{QSCORE_PATH}scores.csv", sep=";", low_memory=False)
cities = np.sort(data_df["city"].unique()).tolist()
data_df["image_path"] = f"{IMAGES_PATH}" + data_df["image_path"]
data_df

##### Convert classes and colors file

In [ ]:
from py.datasets import UrbanPhysicalDisorder

uss = UrbanPhysicalDisorder(data_path=DATA_PATH)
uss.generate_dataset(dataset="ade20k")

objects_df = uss.get_urban_street_categories()
objects_df

### Zero-shot Segmenter

In [ ]:
from py.models.segmentation import ImageSegmenter

ts = ImageSegmenter(model_name=MODEL_NAME, model_path=MODEL_PATH)
ts.to_device(device)
ts.eval()
ts.model_zoo()
ts.print_trainable_parameters()
ts.get_model()

In [ ]:
columns_to_keep = ["image_id", "seg_image_path", "seg_overlay_image_path", "mask_path", "ratio_path"]

In [ ]:
%%time
segment_df = pd.DataFrame()
for city in cities:
    print("Evaluating city", city)
    OUT_DIR = f"{SEGMENT_DIR}/{city}/"
    verifyDir(OUT_DIR)
    verifyDir(f"{OUT_DIR}/masks/")
    verifyDir(f"{OUT_DIR}/ratios/")
    verifyDir(f"{OUT_DIR}/segmented_images/")
    verifyDir(f"{OUT_DIR}/segmented_images_overlay/")

    city_df = data_df[ (data_df["city"]==city) ][["image_id", "image_path"]].copy()
    country = data_df[ (data_df["city"]==city) ]["country"].unique()[0]
    city_df.reset_index(drop=True, inplace=True)
    city_segment_df = pd.DataFrame()

    for index, row in tqdm(city_df.iterrows()):
        img_path = row["image_path"]
        out_name = img_path.split("/")[-1].replace(".JPG", "")

        if verifyFile(f"{OUT_DIR}/masks/{out_name}.pkl") and verifyFile(f"{OUT_DIR}/ratios/{out_name}.csv") and verifyFile(f"{OUT_DIR}/segmented_images/{out_name}.png") and verifyFile(f"{OUT_DIR}/segmented_images_overlay/{out_name}.png"):
            ratio_df = pd.read_csv(f"{OUT_DIR}/ratios/{out_name}.csv", sep=";", low_memory=False)
        else:
            image = Image.open(img_path).convert("RGB")

            ratio_df, masks, seg_img, seg_overlay_img = ts.zeroshot_segmentation(image, objects_df, alpha=0.6)

            # saving image
            seg_img.save(f"{OUT_DIR}/segmented_images/{out_name}.png")
            seg_overlay_img.save(f"{OUT_DIR}/segmented_images_overlay/{out_name}.png")
            #cv2.imwrite(output_seg, np.array(seg_img))
            #cv2.imwrite(output_seg_overlay, np.array(seg_overlay_img))

            # masks
            joblib.dump(masks, f"{OUT_DIR}/masks/{out_name}.pkl")

            # ratios
            ratio_df.to_csv(f"{OUT_DIR}/ratios/{out_name}.csv", sep=";", index=False)

        df_pivot = uss.parse_ratios(out_name, ratio_df)
        df_pivot["seg_image_path"] = f"{city}/segmented_images/{out_name}.png"
        df_pivot["seg_overlay_image_path"] = f"{city}/segmented_images_overlay/{out_name}.png"
        df_pivot["mask_path"] = f"{city}/masks/{out_name}.pkl"
        df_pivot["ratio_path"] = f"{city}/ratios/{out_name}.csv"
        
        city_segment_df = pd.concat([city_segment_df, df_pivot], ignore_index=True)
        city_segment_df = city_segment_df[columns_to_keep + [col for col in city_segment_df.columns if col not in columns_to_keep]].copy()
        city_segment_df.fillna(0, inplace=True)

    city_df = pd.merge(city_df, city_segment_df, on="image_id", how="inner").copy()
    city_df.fillna(0, inplace=True)
    city_df.drop(columns=["image_path"], inplace=True)
    city_df.to_csv(f"{OUT_DIR}/segmentations.csv", sep=";", index=False)
        
    segment_df = pd.concat([segment_df, city_df], ignore_index=True)
    segment_df.fillna(0, inplace=True)

In [ ]:
segment_df

In [ ]:
np.sort(segment_df.columns)

In [ ]:
segment_df.to_csv(f"{SEGMENT_DIR}/segmentations.csv", sep=";", index=False)